In [0]:
%sql
create catalog if not exists autoloader;
create schema if not exists autoloader.bronze;
use catalog autoloader;


In [0]:
ipputfile="s3://streamautoloader/input"
schemalocation="s3://streamautoloader/schema"
checkpointlocation="s3://streamautoloader/checkpoint"


In [0]:
from delta.tables import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
from pyspark.sql.window import Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit, unix_timestamp, from_unixtime, to_timestamp, to_date, to_timestamp, to_date, to_timestamp, to_date, to_timestamp, to_date, to_timestamp, to_date

In [0]:
inputfile = "s3://streamautoloader/input/"
schemalocation = "s3://streamautoloader/schema/"
checkpointlocation = "s3://streamautoloader/checkpoint/"
outputlocation = "s3://streamautoloader/output/"

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schemalocation)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .load(inputfile)
)

df.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpointlocation) \
    .trigger(availableNow=True) \
    .outputMode("append") \
    .toTable("autoloader.bronze.sales")

In [0]:
%sql
select * from autoloader.bronze.sales

In [0]:
from pyspark.sql.functions import col, to_date

bronze_df = spark.read.table("autoloader.bronze.sales")

silver_df = (
    bronze_df
    .filter(col("_rescued_data").isNull())   # remove bad data
    .withColumn("amount", col("amount").cast("int"))
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
    .drop("_rescued_data")

)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("autoloader.silver.sales_clean")

In [0]:
%sql
SELECT * FROM autoloader.silver.sales_clean;

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS autoloader.gold")

In [0]:
silver_df = spark.read.table("autoloader.silver.sales_clean")

In [0]:
from pyspark.sql.functions import sum

gold_df = (
    silver_df
    .groupBy("city", "order_date")
    .agg(sum("amount").alias("total_sales"))
)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS autoloader.gold")

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("autoloader.gold.sales_summary")

In [0]:
%sql
SELECT * FROM autoloader.gold.sales_summary;

In [0]:
display(dbutils.fs.ls("s3://streamautoloader/input/"))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window = Window.partitionBy("city").orderBy(desc("total_sales"))

top_products = (
    gold_df
    .withColumn("rank", row_number().over(window))
    .filter("rank = 1")
)

In [0]:
top_products.show()

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS autoloader.gold.customer_scd (
    order_id STRING,
    product STRING,
    amount INT,
    city STRING,
    order_date DATE,
    is_current STRING,
    start_date DATE,
    end_date DATE
)
USING DELTA
""")

In [0]:
source_df = spark.read.table("autoloader.silver.sales_clean")

In [0]:
from pyspark.sql.functions import current_date, lit

source_df = (
    source_df
    .withColumn("is_current", lit("Y"))
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast("date"))
)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "autoloader.gold.customer_scd")

(
    target.alias("t")
    .merge(
        source_df.alias("s"),
        "t.order_id = s.order_id AND t.is_current = 'Y'"
    )
    .whenMatchedUpdate(
        condition="t.city <> s.city",
        set={
            "is_current": "'N'",
            "end_date": "current_date()"
        }
    )
    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "product": "s.product",
            "amount": "s.amount",
            "city": "s.city",
            "order_date": "s.order_date",
            "is_current": "s.is_current",
            "start_date": "s.start_date",
            "end_date": "s.end_date"
        }
    )
    .execute()
)

In [0]:
%sql
SELECT * FROM autoloader.gold.customer_scd
ORDER BY order_id, start_date;

In [0]:
silver_stream_df = (
    spark.readStream
    .table("autoloader.silver.sales_clean")
)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_date, lit

def scd2_merge(microBatchDF, batchId):

    # Add SCD columns
    df = (
        microBatchDF
        .withColumn("is_current", lit("Y"))
        .withColumn("start_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
    )

    target = DeltaTable.forName(spark, "autoloader.gold.customer_scd")

    (
        target.alias("t")
        .merge(
            df.alias("s"),
            "t.order_id = s.order_id AND t.is_current = 'Y'"
        )
        .whenMatchedUpdate(
            condition="t.city <> s.city",
            set={
                "is_current": "'N'",
                "end_date": "current_date()"
            }
        )
        .whenNotMatchedInsert(
            values={
                "order_id": "s.order_id",
                "product": "s.product",
                "amount": "s.amount",
                "city": "s.city",
                "order_date": "s.order_date",
                "is_current": "s.is_current",
                "start_date": "s.start_date",
                "end_date": "s.end_date"
            }
        )
        .execute()
    )

In [0]:
(
    silver_stream_df.writeStream
    .foreachBatch(scd2_merge)
    .option("checkpointLocation", "s3://streamautoloader/scd_checkpoint/")
    .trigger(availableNow=True)
    .start()
)

In [0]:
%sql
SELECT * FROM autoloader.gold.customer_scd where  order_id
ORDER BY order_id, start_date; 

In [0]:
%sql
SELECT order_id, count(*) 
FROM autoloader.gold.customer_scd
WHERE is_current = 'Y'
GROUP BY order_id
HAVING count(*) != 1;